# Modul 07: Optimierung und lineare Regression mit NumPy | Lösungen

## Überblick

Sie machen Verlustlandschaften sichtbar, implementieren Raster- und Gradientenabstieg und trainieren anschließend ein lineares Regressionsmodell vollständig mit NumPy. Lernrate, Skalierung, Abbruch, Validierung, geschlossene Lösung, Residuen und eine polynomiale Erweiterung werden verglichen.

**Zugehörige Vorlesungen**

- **Optimierung verstehen**
- **Regression mit NumPy**

## Lernziele

Nach der Bearbeitung können Sie:

- Verlustfunktionen und lokale Steigungen für einzelne Parameter berechnen und visualisieren.
- Gradientenabstieg mit Lernrate, Epochen und Abbruchbedingung implementieren und diagnostizieren.
- Gewicht und Bias einer linearen Regression trainieren und mit Baseline, geschlossener Lösung und Polynommodell vergleichen.

## Geprüfte Fähigkeiten

- Rasterauswertung, numerischer Gradient und Konvergenzkurven
- vektorisierte MSE-Gradienten für y = w*x + b
- Residualanalyse, Skalierung und kontrollierter Modellvergleich

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Ein kleiner synthetischer Datensatz simuliert den Zusammenhang zwischen Maschinenalter und Energieverbrauch. Die Daten werden reproduzierbar erzeugt und in Training, Validierung und Test geteilt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

x_gesamt = np.linspace(0, 10, 90)
y_gesamt = 4.2 * x_gesamt + 7.5 + rng.normal(0, 3.0, size=x_gesamt.size)

# Chronologisch geordnete Teilung ist hier nur eine kontrollierte Aufgabenkonstruktion.
x_train = x_gesamt[:55]
y_train = y_gesamt[:55]
x_valid = x_gesamt[55:72]
y_valid = y_gesamt[55:72]
x_test = x_gesamt[72:]
y_test = y_gesamt[72:]

print("Einrichtung abgeschlossen.")
print("Train/Valid/Test:", len(x_train), len(x_valid), len(x_test))

### Aufgabe 1: Verlust eines einzelnen Parameters im Raster untersuchen

Nehmen Sie zunächst den Bias als fest `b = 7.5` an und optimieren Sie nur die Steigung `w`:

1. Implementieren Sie `mse_fuer_steigung(w, x, y, b)`.
2. Berechnen Sie den Trainingsverlust für 101 Steigungen zwischen 0 und 8.
3. Bestimmen Sie den besten Rasterwert.
4. Visualisieren Sie Verlust gegen Steigung und markieren Sie das Minimum.
5. Erklären Sie den Einfluss der Rasterfeinheit.

In [ ]:
def mse_fuer_steigung(w, x, y, b=7.5):
    """Berechnet den mittleren quadratischen Fehler für eine feste Steigung."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def mse_fuer_steigung(w, x, y, b=7.5):
    """Berechnet den mittleren quadratischen Fehler für eine feste Steigung."""
    vorhersage = w * x + b
    fehler = vorhersage - y
    return np.mean(fehler ** 2)


steigungen = np.linspace(0.0, 8.0, 101)
verluste = np.array([mse_fuer_steigung(w, x_train, y_train) for w in steigungen])
bester_index = int(np.argmin(verluste))
beste_steigung = steigungen[bester_index]

print(f"Beste Rastersteigung: {beste_steigung:.3f}")
print(f"Kleinster Rasterverlust: {verluste[bester_index]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steigungen, verluste)
ax.scatter([beste_steigung], [verluste[bester_index]], s=80, label="Rasterminimum")
ax.set_title("Verlustlandschaft für die Steigung")
ax.set_xlabel("Steigung w")
ax.set_ylabel("Trainings-MSE")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Ein feineres Raster kann das Minimum genauer eingrenzen, benötigt aber mehr Verlustberechnungen. Es bleibt außerdem auf den gewählten Suchbereich beschränkt und wird bei mehreren Parametern schnell unpraktisch.

### Aufgabe 2: Lokale Steigung numerisch prüfen

1. Implementieren Sie einen zentralen numerischen Gradienten für `mse_fuer_steigung`.
2. Leiten Sie den analytischen Gradienten nach `w` her und implementieren Sie ihn.
3. Vergleichen Sie beide Gradienten bei `w = 2.0`, `4.0` und `6.0`.
4. Interpretieren Sie Vorzeichen und Betrag.

In [ ]:
def numerischer_gradient(w, x, y, b=7.5, epsilon=1e-5):
    pass


def analytischer_gradient_w(w, x, y, b=7.5):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def numerischer_gradient(w, x, y, b=7.5, epsilon=1e-5):
    # Zentrale Differenz nutzt Werte links und rechts und ist genauer als eine einseitige Differenz.
    loss_plus = mse_fuer_steigung(w + epsilon, x, y, b)
    loss_minus = mse_fuer_steigung(w - epsilon, x, y, b)
    return (loss_plus - loss_minus) / (2 * epsilon)


def analytischer_gradient_w(w, x, y, b=7.5):
    # Für MSE = mean((w*x+b-y)^2) ergibt die Kettenregel 2*mean((pred-y)*x).
    fehler = (w * x + b) - y
    return 2.0 * np.mean(fehler * x)


vergleich_gradient = []
for w in [2.0, 4.0, 6.0]:
    num = numerischer_gradient(w, x_train, y_train)
    ana = analytischer_gradient_w(w, x_train, y_train)
    vergleich_gradient.append(
        {"w": w, "numerisch": num, "analytisch": ana, "absolute_Differenz": abs(num - ana)}
    )

display(pd.DataFrame(vergleich_gradient))

> **Musterantwort und Interpretation**
>
> Ein negativer Gradient bedeutet, dass der Verlust zunimmt, wenn w lokal kleiner wird, beziehungsweise abnimmt, wenn w größer wird. Der Gradientenabstieg subtrahiert den Gradienten und erhöht w daher in diesem Fall. Der Betrag zeigt die lokale Empfindlichkeit.

### Aufgabe 3: Gradientenabstieg auf einer einfachen Funktion diagnostizieren

Optimieren Sie die Funktion `f(p) = (p - 3)^2 + 2`:

1. Implementieren Sie Gradientenabstieg mit Verlaufsliste und Toleranz für `abs(gradient) < toleranz`.
2. Vergleichen Sie Lernraten 0,05, 0,30 und 1,10 bei Startwert -4.
3. Visualisieren Sie die Verlustverläufe.
4. Dokumentieren Sie Konvergenz, Oszillation oder Divergenz.

In [ ]:
def einfache_funktion(p):
    return (p - 3.0) ** 2 + 2.0


def einfacher_gradient(p):
    return 2.0 * (p - 3.0)


def gradientenabstieg_einfach(start, lernrate, max_epochen=50, toleranz=1e-6):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def gradientenabstieg_einfach(start, lernrate, max_epochen=50, toleranz=1e-6):
    p = float(start)
    verlauf = []

    for epoche in range(max_epochen):
        verlust = einfache_funktion(p)
        grad = einfacher_gradient(p)
        verlauf.append({"Epoche": epoche, "Parameter": p, "Verlust": verlust, "Gradient": grad})

        if abs(grad) < toleranz:
            break

        # Der Schritt erfolgt entgegen der lokalen Steigung.
        p = p - lernrate * grad

    return p, pd.DataFrame(verlauf)


ergebnisse = {}
fig, ax = plt.subplots(figsize=(8, 5))
for lr in [0.05, 0.30, 1.10]:
    parameter, historie = gradientenabstieg_einfach(-4.0, lr)
    ergebnisse[lr] = (parameter, historie)
    ax.plot(historie["Epoche"], historie["Verlust"], marker="o", markersize=3, label=f"lr={lr}")
    print(f"Lernrate {lr}: letzter Parameter {parameter:.4f}, letzter Verlust {historie['Verlust'].iloc[-1]:.4f}")

ax.set_title("Lernrate und Konvergenz")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")
ax.set_yscale("log")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Der Schritt kann über das Minimum hinaus springen und die Entfernung zum Minimum sogar vergrößern. Bei dieser quadratischen Funktion führt eine Lernrate von 1,10 zu einem Multiplikationsfaktor mit Betrag größer als 1 für den Parameterfehler, wodurch die Oszillation anwächst.

### Aufgabe 4: Lineare Regression mit NumPy trainieren

Implementieren Sie eine Funktion `trainiere_lineare_regression`, die `w` und `b` mit Gradientenabstieg lernt:

- Vorhersage `w*x + b`,
- MSE-Verlust,
- Gradienten für `w` und `b`,
- Trainings- und Validierungsverlust pro Epoche,
- optionales Early Stopping, wenn sich der Validierungsverlust über 30 Epochen nicht verbessert.

Trainieren Sie auf skaliertem `x`, damit die Optimierung stabil ist. Speichern Sie Skalierungsparameter nur aus dem Training.

In [ ]:
def trainiere_lineare_regression(
    x_train_skaliert,
    y_train,
    x_valid_skaliert,
    y_valid,
    lernrate=0.05,
    max_epochen=2000,
    geduld=30,
):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Skalierungsparameter stammen ausschließlich aus dem Training.
x_mittel = x_train.mean()
x_std = x_train.std()
x_train_s = (x_train - x_mittel) / x_std
x_valid_s = (x_valid - x_mittel) / x_std
x_test_s = (x_test - x_mittel) / x_std


def trainiere_lineare_regression(
    x_train_skaliert,
    y_train,
    x_valid_skaliert,
    y_valid,
    lernrate=0.05,
    max_epochen=2000,
    geduld=30,
):
    w = 0.0
    b = float(np.mean(y_train))
    historie = []
    bester_valid_loss = np.inf
    beste_parameter = (w, b)
    epochen_ohne_verbesserung = 0

    for epoche in range(max_epochen):
        train_pred = w * x_train_skaliert + b
        train_fehler = train_pred - y_train
        train_loss = np.mean(train_fehler ** 2)

        # Vektorisierte Gradienten des mittleren quadratischen Fehlers.
        grad_w = 2.0 * np.mean(train_fehler * x_train_skaliert)
        grad_b = 2.0 * np.mean(train_fehler)

        w -= lernrate * grad_w
        b -= lernrate * grad_b

        valid_pred = w * x_valid_skaliert + b
        valid_loss = np.mean((valid_pred - y_valid) ** 2)
        historie.append((epoche, train_loss, valid_loss, w, b))

        if valid_loss < bester_valid_loss - 1e-8:
            bester_valid_loss = valid_loss
            beste_parameter = (w, b)
            epochen_ohne_verbesserung = 0
        else:
            epochen_ohne_verbesserung += 1

        if epochen_ohne_verbesserung >= geduld:
            break

    historie_df = pd.DataFrame(
        historie,
        columns=["Epoche", "Train_MSE", "Valid_MSE", "w_skaliert", "b_skaliert"],
    )
    return beste_parameter[0], beste_parameter[1], historie_df


w_s, b_s, trainingshistorie = trainiere_lineare_regression(
    x_train_s, y_train, x_valid_s, y_valid
)
print(f"Gelernte Parameter im skalierten Raum: w={w_s:.3f}, b={b_s:.3f}")
print("Ausgeführte Epochen:", len(trainingshistorie))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(trainingshistorie["Epoche"], trainingshistorie["Train_MSE"], label="Training")
ax.plot(trainingshistorie["Epoche"], trainingshistorie["Valid_MSE"], label="Validierung")
ax.set_title("Trainings- und Validierungsverlust")
ax.set_xlabel("Epoche")
ax.set_ylabel("MSE")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Die Skalierung ist ein gelernter Vorverarbeitungsschritt. Würden Validierungs- oder Testwerte einbezogen, flösse Information über deren Verteilung in das Training ein. Die späteren Splits müssen mit den unveränderten Trainingsparametern transformiert werden.

### Aufgabe 5: Geschlossene Lösung, Baseline und Residuen vergleichen

1. Berechnen Sie Testvorhersagen des Gradientenmodells.
2. Berechnen Sie eine Mittelwert-Baseline aus `y_train`.
3. Bestimmen Sie die geschlossene lineare Lösung mit einer Designmatrix aus Einsen und `x_train` und `np.linalg.pinv`.
4. Trainieren Sie zusätzlich `sklearn.linear_model.LinearRegression` als Kontrollmodell.
5. Vergleichen Sie Test-MSE und stellen Sie Vorhersagegeraden sowie Residuen dar.

In [ ]:
# w_s, b_s und x_test_s stammen aus Aufgabe 4.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Gradientenmodell im skalierten Merkmalsraum.
pred_gradient = w_s * x_test_s + b_s

# Einfache Baseline sagt für jeden Testfall den Trainingsmittelwert voraus.
pred_baseline = np.full_like(y_test, y_train.mean(), dtype=float)

# Geschlossene Lösung mit Designmatrix [x, 1].
design_train = np.column_stack([x_train, np.ones_like(x_train)])
parameter_closed = np.linalg.pinv(design_train) @ y_train
w_closed, b_closed = parameter_closed
pred_closed = w_closed * x_test + b_closed

# scikit-learn dient nur als unabhängige Kontrolle derselben linearen Modellklasse.
sk_modell = LinearRegression()
sk_modell.fit(x_train.reshape(-1, 1), y_train)
pred_sklearn = sk_modell.predict(x_test.reshape(-1, 1))

vergleich = pd.DataFrame(
    {
        "Modell": ["Mittelwert-Baseline", "Gradientenabstieg", "Geschlossene Lösung", "scikit-learn"],
        "Test_MSE": [
            mean_squared_error(y_test, pred_baseline),
            mean_squared_error(y_test, pred_gradient),
            mean_squared_error(y_test, pred_closed),
            mean_squared_error(y_test, pred_sklearn),
        ],
    }
)
display(vergleich.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x_test, y_test, label="Testdaten")
axes[0].plot(x_test, pred_gradient, label="Gradientenmodell")
axes[0].plot(x_test, pred_closed, linestyle="--", label="geschlossene Lösung")
axes[0].set_title("Vorhersagen auf Testdaten")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend()

residuen = pred_gradient - y_test
axes[1].axhline(0, linewidth=1)
axes[1].scatter(pred_gradient, residuen)
axes[1].set_title("Residuen des Gradientenmodells")
axes[1].set_xlabel("Vorhersage")
axes[1].set_ylabel("Vorhersage minus Istwert")
plt.tight_layout()
plt.show()

> **Musterantwort und Interpretation**
>
> Ein systematisches gekrümmtes Muster deutet darauf hin, dass die lineare Gerade die Datenstruktur nicht vollständig erfasst. Möglicherweise fehlt ein nichtlinearer Term, ein wichtiges Merkmal oder eine Transformation. Residuen sollten idealerweise ohne klares Muster um null streuen.

### Aufgabe 6: Integrationsaufgabe: polynomiale Erweiterung kontrollieren

Erzeugen Sie einen neuen Datensatz mit leichter Krümmung `y = 2 + 1.5*x + 0.35*x^2 + Rauschen` und vergleichen Sie:

1. Mittelwert-Baseline,
2. lineares Modell,
3. Polynommodell zweiten Grades.

Verwenden Sie denselben Train/Test-Split für alle Modelle. Berechnen Sie Test-MSE, visualisieren Sie die Kurven und diskutieren Sie, warum ein sehr hoher Polynomgrad riskant wäre.

In [ ]:
rng_poly = np.random.default_rng(RANDOM_SEED)
x_poly = np.linspace(-3, 5, 100)
y_poly = 2 + 1.5 * x_poly + 0.35 * x_poly**2 + rng_poly.normal(0, 1.2, size=x_poly.size)

x_poly_train = x_poly[:75]
y_poly_train = y_poly[:75]
x_poly_test = x_poly[75:]
y_poly_test = y_poly[75:]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Baseline wird ausschließlich aus Trainingszielen bestimmt.
poly_baseline = np.full_like(y_poly_test, y_poly_train.mean(), dtype=float)

lineares_modell = LinearRegression()
lineares_modell.fit(x_poly_train.reshape(-1, 1), y_poly_train)
linear_pred = lineares_modell.predict(x_poly_test.reshape(-1, 1))

# PolynomialFeatures erzeugt [1, x, x²]. include_bias=False vermeidet eine doppelte Bias-Spalte.
poly_transformer = PolynomialFeatures(degree=2, include_bias=False)
X_poly_train = poly_transformer.fit_transform(x_poly_train.reshape(-1, 1))
X_poly_test = poly_transformer.transform(x_poly_test.reshape(-1, 1))
poly_modell = LinearRegression()
poly_modell.fit(X_poly_train, y_poly_train)
poly_pred = poly_modell.predict(X_poly_test)

poly_vergleich = pd.DataFrame(
    {
        "Modell": ["Mittelwert", "linear", "Polynom Grad 2"],
        "Test_MSE": [
            mean_squared_error(y_poly_test, poly_baseline),
            mean_squared_error(y_poly_test, linear_pred),
            mean_squared_error(y_poly_test, poly_pred),
        ],
    }
)
display(poly_vergleich.round(3))

x_plot = np.linspace(x_poly.min(), x_poly.max(), 250)
linear_plot = lineares_modell.predict(x_plot.reshape(-1, 1))
poly_plot = poly_modell.predict(poly_transformer.transform(x_plot.reshape(-1, 1)))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x_poly_train, y_poly_train, alpha=0.6, label="Training")
ax.scatter(x_poly_test, y_poly_test, alpha=0.8, label="Test")
ax.plot(x_plot, linear_plot, label="linear")
ax.plot(x_plot, poly_plot, label="Grad 2")
ax.set_title("Lineares und polynomiales Modell")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Die Auswahl des Polynomgrads ist eine Modellentscheidung und gehört auf Trainings- und Validierungsdaten. Wird der Test mehrfach zur Auswahl genutzt, passt sich der gesamte Entwicklungsprozess indirekt an den Test an, und dessen spätere Leistung ist keine unabhängige Schätzung mehr.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?